# 03 · Gold e qualidade
Monta as tabelas de consumo e roda as checagens. Se alguma falhar, o notebook para com erro (o pipeline do Fabric marca a execução como falha).

In [ ]:
# Pré-requisito: a pasta src/medallion do repositório foi enviada para
# Files/medallion_src/medallion no Lakehouse padrão (veja o README, passo 2).
import sys
sys.path.append("/lakehouse/default/Files/medallion_src")

import pandas as pd
from medallion import bronze, silver, gold, quality, fabric, rules
from medallion.generate import generate_raw

RAW = f"{fabric.LAKEHOUSE_FILES}/raw"
HOJE = "2025-08-01"  # data de referência para detectar datas futuras

In [ ]:
s = {t: fabric.read_delta(spark, f"silver_{t}") for t in ("escolas", "turmas", "alunos", "frequencia")}

por_aluno = gold.frequencia_aluno(s["frequencia"], s["alunos"], s["turmas"], s["escolas"])
g = {
    "frequencia_mensal_turma": gold.frequencia_mensal_turma(s["frequencia"], s["alunos"], s["turmas"], s["escolas"]),
    "frequencia_aluno": por_aluno,
    "resumo_escola": gold.resumo_escola(por_aluno),
}
for nome, df in g.items():
    fabric.save_delta(spark, df, f"gold_{nome}")
    print(f"gold_{nome}: {len(df)} linhas")

In [ ]:
relatorio = quality.run_checks(s, g)
fabric.save_delta(spark, relatorio, "qualidade")
display(relatorio)

falhas = relatorio[relatorio["status"] != "OK"]
assert falhas.empty, f"{len(falhas)} checagem(ns) de qualidade falharam"